# Notebook 2 — CV by Variable Type

Holding geography fixed at **census tract**, we compare CV across variable types: population, income, poverty, and small subgroup cells (Black 65+).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'JL_Analysis':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'analysis' / 'JL_Analysis'
sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import VARIABLES, build_cv_long

In [ ]:
cv_tract = build_cv_long('tract').dropna(subset=['cv'])
summary = (
    cv_tract.groupby(['variable_group', 'variable'])
    .agg(n=('cv', 'count'), median_cv=('cv', 'median'), p90_cv=('cv', lambda s: s.quantile(0.9)))
    .sort_values('median_cv', ascending=False)
)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

group_order = ['population', 'income', 'poverty', 'subgroup']
group_data = [cv_tract.loc[cv_tract['variable_group'] == g, 'cv'].values for g in group_order]
axes[0].boxplot(group_data, tick_labels=group_order, showfliers=False)
axes[0].set_yscale('log')
axes[0].set_title('Tract-level CV by variable group')
axes[0].set_xlabel('Variable group')
axes[0].set_ylabel('CV (log scale)')

for group, color in zip(group_order, ['#2c5f8a', '#4a8fb8', '#7eb8d8', '#f4a261']):
    sub = cv_tract[cv_tract['variable_group'] == group]
    axes[1].scatter(sub['variable'], sub['cv'], alpha=0.35, s=8, label=group, c=color)
axes[1].set_yscale('log')
axes[1].set_title('Tract-level CV by individual variable')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Group', fontsize=8)
plt.tight_layout()
plt.show()

### Takeaway

Small subgroup counts (Black 65+ cells) show the highest tract-level CV — often an order of magnitude above population or income. Poverty sits in the middle; total population is the most stable at tract level.